In [142]:
basis_sets_dir = "./basis-sets"

particle_properties_file = "particle-properties.json"

truncate_e = 0

mtx_elmt_threshold = 1e-7

mol_xyz = 'mols/H22.xyz'
spin_file = 'H2.spin'



import numpy as np
import scipy as sp
import json
import itertools
import argparse
import time
import math

from scipy.sparse import coo_matrix, csr_matrix
from scipy.linalg import block_diag

from pyscf import gto, scf
from pyscf.lo import orth

from gbasis.wrappers import from_pyscf
from gbasis.parsers import parse_nwchem
from gbasis.parsers import make_contractions

from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral



# Load properties of all possible particles (spin, fermion/boson, mass, charge, etc)
with open(particle_properties_file, "r") as file:
    particle_properties = json.load(file)

# Build molecule for PySCF
mol = gto.Mole()
mol.atom = mol_xyz
mol.basis = e_basis_set
mol.build()

mol_zs = mol.atom_charges()
mol_symbs = [mol.atom_symbol(i) for i in range(mol.natm)] # Atomic symbols
mol_coords = mol.atom_coords()

# Run restricted Hartree Fock to get better orbitals (I might truncate the highest energy ones)
hf = scf.RHF(mol).run() # TODO: apparently there might be better choices of orbital to allow for truncations (FNO). look into?

# Load basis dictionary (atomic orbitals) for nuclear orbitals
n_basis_dict = parse_nwchem(basis_sets_dir + '/nuclear/' + n_basis_set + '.nw')

# Construct a dictionary of all the particle types that will be in our calculation, along with their orbitals and info like spin.
# Note: we will use the order of the dictionary. Python 3.7+ guarantees when we iterate, the dictionary will be ordered according to when the elements were added.
particles = {}

for i in range(mol.natm):
    symb = mol.atom_symbol(i)
    # If this is a particle type (nucleus) we haven't seen before
    if symb not in particles:
        # Add it to particle list
        particles[symb] = {}
        particles[symb]['coords'] = []
        particles[symb]['count'] = 0

    # Add its coordinates to the list
    particles[symb]['coords'].append(mol_coords[i])

    particles[symb]['count'] += 1

for symb in particles:
    # GBasis wants coords as numpy array
    particles[symb]['coords'] = np.array(particles[symb]['coords'])

    # Construct a basis w/ GBasis for each of the nuclear particles
    particles[symb]['basis'] = make_contractions(n_basis_dict, # Basis of (nuclear) AOs to use
                                                 [symb] * particles[symb]['count'], # Types of atoms (all the same)
                                                 particles[symb]['coords'], # Coordinates
                                                 coord_types='cartesian')

    # Transform to orthonormal orbitals
    overlap = overlap_integral(particles[symb]['basis'])
    particles[symb]['transform'] = orth.lowdin(overlap) # Symmetric orthonormalization of AOs

    # Number of spatial orbitals
    particles[symb]['no_spatial_orbitals'] = overlap.shape[0]

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['properties'] = particle_properties[symb]

# Add electrons to our particle list
particles['e'] = {}
particles['e']['basis'] = from_pyscf(mol) # Gbasis set of GTOs (gaussian type orbitals) for electronic particles
particles['e']['transform'] = hf.mo_coeff.T # transform to MOs that will be used for calculation
particles['e']['count'] = mol.nelectron  # Get number of electrons from PySCF, truncate off 10
particles['e']['no_spatial_orbitals'] = hf.mo_coeff.shape[0] - truncate_e

idx = 0

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['idx'] = idx
    idx += 1

    particles[symb]['properties'] = particle_properties[symb]

    # Number of spin orbitals, since we now have the particle's spin
    particles[symb]['no_spin_orbitals'] = particles[symb]['no_spatial_orbitals'] * particles[symb]['properties']['spin']

# Read in file containing the count of particles in each spin state (alpha, beta, etc) for each particle
with open(spin_file, 'r', encoding='utf-8') as file:
    while True:
        line = file.readline()
        if not line:
            break  # End of file reached

        split = line.strip().split()
        particles[split[0]]['occs'] = [int(s) for s in split[1:]]

# Amount of particles we have
particle_types = len(particles)

# List of particle names for easy indexing
particle_names = [symb for symb in particles]

# TODO: I should probably just do all the integrals at once by combining the bases of all particles.

# Our basis in particles[symb]['basis'] is a basis of spatial orbitals, so the integrals will be between spatial oritals.
# This class will take a matrix (or 4D array for 2-body interactions) of these spatial integrals and give us a matrix that indexes spin orbitals
# We use the convention defined below for indexing spin orbitals
#
# This class ASSUMES 2-body integrals are in CHEMIST'S NOTATION:
# data[i,j,k,l] = \int dr_1 dr_2 \chi_i^*(r_1) \chi_j(r_1) \hat{O}_2 \chi_k^*(r_2) \chi_l(r_2)
class IntegralSpinWrapper:
    # data: table of integrals (2D array for 1-body, 4D array for 2-body). Should be a numpy array, which it will be for integrals from GBasis
    # spin: number of spin states if 1-body, or 2-tuple of the number of spin states for each particle if 2-body
    def __init__(self, data, spin):
        self.data = data
        self.spin = spin

        self.is_two_body = len(data.shape) == 4

    def __getitem__(self, index):
        # If one-body interaction
        if not self.is_two_body:
            # Spin states must be equal.
            if index[0] % self.spin == index[1] % self.spin:
                return self.data[int(index[0] / self.spin), int(index[1] / self.spin)]

            # Otherwise by orthonormality the integral is 0
            else:
                return 0

        # If two-body interaction
        else:
            # Spin states must be equal for both coordinates:
            if (index[0] % self.spin[0] == index[1] % self.spin[0]) and (index[2] % self.spin[1] == index[3] % self.spin[1]):
                return self.data[int(index[0]/self.spin[0]), int(index[1]/self.spin[0]), int(index[2]/self.spin[1]), int(index[3]/self.spin[1])]

            # Otherwise by orthonormality the integral is 0
            else:
                return 0

# For full FCI, we will construct the Hamiltonian in the subspace of all states with only the correct particle numbers
# A basis for this space is constructed from N-particle determinants/permanents of the one-particle basis states for correct N

# Total number of states in this space
total_states = 1

# For the Hamiltonian matrix, we will index the states as follows:
# Let N_i be the number of states for the i-th particle
# The state formed from the c_0 - th particle 1 permanent/determinant, c_1 - th particle 2 perminant/determinant, etc (c_i's zero indexed)
# Will get the index (c_0) + (c_1 * N_0) + (c_2 * N_1 * N_0) + (c_3 * N_2 * N_1 * N_0) + ...

# This works basically like a numeral base system where each digit has a different base (each digit is a particle).
# This array will contain the bases for each particle: [1, N_0, N_1 * N_0, ...]
bases = []


# Given a certain N-particle state index, construct an array of the indices of all states differing by d1one-particle states, combined with info on which states are different and what the parity factor is after aligning up all the common states
# gamma^IJ_ij = parity
# i = occupy, j = deocccupy
# J = input state, I = output state
def get_diff_states(occupied, addr_array):
    no_orbitals = addr_array.shape[1]

    # Indices of the occupied & unoccupied one-particle states for this N-particle state
    unoccupied = [i for i in range(no_orbitals) if i not in occupied]


    new_states = []

    state_idx = get_addr(occupied, addr_array)

    for deoccupy_idx, deoccupy in enumerate(occupied):
        for occupy in unoccupied:
            # Construct new state
            new_occupied = occupied.copy()

            new_occupied[deoccupy_idx] = occupy
            new_occupied.sort()


            # States in common
            common = [i for i in occupied if i != deoccupy]

            # Get index of this new state
            new_state_idx = get_addr(new_occupied, addr_array)

            # How many permutations to align the old state and new state's determinants?
            perms = sum([min(deoccupy, occupy) < c < max(deoccupy, occupy) for c in common])

            # Fermion exchange parity from aligning the determinants
            parity = 1 if perms % 2 == 0 else -1

            # Package all this info together
            new_states.append((new_state_idx, occupy, deoccupy, parity))

        # Now for replacements that give us the same state, E_ii:
        new_states.append((state_idx, deoccupy, deoccupy, 1))

    return new_states


import math

def create_addr_array(M, N):
    if N == 0:
        return np.array([[]])

    addr_array = np.zeros((N, M), dtype=int)

    for k in range(N-1):
        for l in range(k, M-N+k+1):
            addr_array[k][l] = sum([
                math.comb(m, N-k-1) - math.comb(m-1, N-k-2) for m in range(M-l, M-k)
            ])

    # second case (k = N-1, or N in the original paper, non-zero indexed)
    for l in range(N-1, M):
        addr_array[N-1][l] = l+1-N

    return addr_array

def get_addr(orbitals, addr_array):
    return sum([addr_array[i,j] for i,j in enumerate(orbitals)])

string_mtx_shape = []

tensor_idx = 0
full_basis_idx = 0

# Construct N-particle states
for symb in particles:
    particle = particles[symb]
    particle['states'] = []
    particle['addr_arrays'] = []
    particle['bases'] = []
    particle['counts'] = []

    no_states = 1


    # Construct all N-particle states for the correct N, in the forms of arrays of 1s and 0s. They will be indexed in the order we get them from itertools
    # States with the same spatial wave function will be grouped together. For example, for 4 spatial orbitals with A and B spin states, the significance of the bits will be
    # Bit number:       12345678
    # Spin:             ABABABAB
    # Spatial orbital:  11223344
    for spin in range(particle['properties']['spin']):
        states = [] # States for this particular particle ("strings" in Handy-Knowles paper)

        occ = particle['occs'][spin] # Number of this particle with this spin
        for indices in itertools.combinations(range(particle['no_spatial_orbitals']), occ):
            #array = [0] * particle['no_spatial_orbitals']
            #for index in indices:
            #    array[index] = 1

            #states.append(array)
            states.append(list(indices))

        particle['states'].append(states)
        particle['bases'].append(no_states)
        particle['counts'].append(len(states))

        particle['addr_arrays'].append(create_addr_array(particle['no_spatial_orbitals'], occ))

        string_mtx_shape.append(len(states))

        no_states *= len(states)
        total_states *= len(states)



    particle['tensor_idx'] = tensor_idx # index in the tensor of strings (think handy knowles alpha beta string)
    particle['full_basis_idx'] = full_basis_idx

    tensor_idx += particle['properties']['spin']
    full_basis_idx += particle['no_spatial_orbitals']

    particle['base'] = total_states

    bases.append(particle['base'])

# Construct single replacements
for symb in particles:
    particle = particles[symb]

    particle['states_r'] = []

    for spin in range(particle['properties']['spin']):
        states_r = [get_diff_states(state, particle['addr_arrays'][spin]) for state in particle['states'][spin]]
        particle['states_r'].append(states_r)

# Combine bases of all the particles (the overlaps between different particles will be nonsensical)
full_basis = tuple(itertools.chain.from_iterable([particles[symb]['basis'] for symb in particles]))

# Full orthonormalization transform for this basis
full_transform = block_diag(*tuple([particles[symb]['transform'] for symb in particles]))

# All coulomb integrals contained in this
full_cmb_int = electron_repulsion_integral(full_basis, notation='chemist', transform=full_transform)

# Now we will multiple these two-electron integrals by the factors coming from the charge and particle statistics
for symb1 in particles:
    particle1 = particles[symb1]
    charge1 = particles[symb1]['properties']['charge']
    fbi1 = particle1['full_basis_idx']
    obtl1 = particle1['no_spatial_orbitals']

    for symb2 in particles:
        particle2 = particles[symb2]
        charge2 = particles[symb2]['properties']['charge']
        fbi2 = particle2['full_basis_idx']
        obtl2 = particle2['no_spatial_orbitals']

        half = 0.5 if symb1 == symb2 else 1

        full_cmb_int[fbi1:fbi1+obtl1, fbi1:fbi1+obtl1, fbi2:fbi2+obtl2, fbi2:fbi2+obtl2] *= charge1 * charge2 * half

print(full_cmb_int)

print("2e ints formed")

def matvec(v):
    print("matvec called")
    C = v.reshape(string_mtx_shape, order='F')

    sigma = np.zeros(string_mtx_shape)
    sigma2 = np.zeros(string_mtx_shape)

    for symb in particles:
        particle = particles[symb]
        tensor_idx = particle['tensor_idx']
        fbi = particle['full_basis_idx']

        mass = particle['properties']['mass']
        spin = particle['properties']['spin']


        fbi = particle['full_basis_idx']
        obtl = particle['no_spatial_orbitals']

        ke_int = kinetic_energy_integral(particle['basis'], transform=particle['transform']) / mass
        cmb_int = full_cmb_int[fbi:fbi+obtl, fbi:fbi+obtl, fbi:fbi+obtl, fbi:fbi+obtl]

        D = np.zeros(tuple(string_mtx_shape) + (particle['no_spatial_orbitals'], particle['no_spatial_orbitals']))
        E = np.zeros(tuple(string_mtx_shape) + (particle['no_spatial_orbitals'], particle['no_spatial_orbitals']))

        # Coulomb interaction between like particles only
        for s in range(spin):
            for state_no, (state, states_r) in enumerate(zip(particle['states'][s], particle['states_r'][s])):
                for state_r_no, i, j, parity in states_r:
                    #print('~~~~~~~~')
                    #print(state_no, state_r_no, '(', i, j, ')', parity)
                    idx_lhs = [slice(None)] * D.ndim
                    idx_rhs = [slice(None)] * C.ndim

                    idx_lhs[tensor_idx + s] = state_no
                    idx_rhs[tensor_idx + s] = state_r_no

                    idx_lhs[-2] = i
                    idx_lhs[-1] = j

                    idx_lhs = tuple(idx_lhs)
                    idx_rhs = tuple(idx_rhs)

                    #print(idx_lhs)
                    #print(idx_rhs)

                    D[idx_lhs] += C[idx_rhs] * parity


        E = np.tensordot(D, cmb_int, axes=([-2, -1], [-2, -1]))


        """
        for i, j, k, l in np.ndindex((obtl, obtl, obtl, obtl)):
            idx_lhs = [slice(None)] * E.ndim
            idx_rhs = [slice(None)] * D.ndim

            idx_lhs[-2] = k
            idx_lhs[-1] = l

            idx_rhs[-2] = i
            idx_rhs[-1] = j

            idx_lhs = tuple(idx_lhs)
            idx_rhs = tuple(idx_rhs)

            E[idx_lhs] += D[idx_rhs] * full_cmb_int[i+fbi, j+fbi, k+fbi, l+fbi]
        """

        for s in range(spin):
            for state_no, (state, states_r) in enumerate(zip(particle['states'][s], particle['states_r'][s])):
                for state_r_no, i, j, parity in states_r:
                    idx_lhs = [slice(None)] * sigma.ndim
                    idx_rhs = [slice(None)] * E.ndim

                    idx_lhs[tensor_idx + s] = state_r_no
                    idx_rhs[tensor_idx + s] = state_no

                    idx_rhs[-2] = i
                    idx_rhs[-1] = j

                    idx_lhs = tuple(idx_lhs)
                    idx_rhs = tuple(idx_rhs)

                    sigma[idx_lhs] += E[idx_rhs] * parity

        for s in range(spin):
            for state_no, (state, states_r) in enumerate(zip(particle['states'][s], particle['states_r'][s])):
                for state_r_no, i, l, parity in states_r:
                    idx_lhs = [slice(None)] * sigma.ndim
                    idx_rhs = [slice(None)] * C.ndim

                    idx_lhs[tensor_idx + s] = state_r_no # state_r_no is I
                    idx_rhs[tensor_idx + s] = state_no # state_no is J

                    elmt = sum(full_cmb_int[i+fbi, j+fbi, j+fbi, l+fbi] for j in range(obtl))

                    idx_lhs = tuple(idx_lhs)
                    idx_rhs = tuple(idx_rhs)

                    sigma[idx_lhs] -= elmt * parity * C[idx_rhs]

        # Kinetic energy
        for s in range(spin):
            #print('~~~~~~~~~~~~~~~~~~~~')
            #print('spin', spin)
            for state_no, (state, states_r) in enumerate(zip(particle['states'][s], particle['states_r'][s])):
                #print('state', state, '-----------')
                for state_r_no, i, j, parity in states_r:
                    idx_lhs = [slice(None)] * sigma.ndim
                    idx_rhs = [slice(None)] * C.ndim

                    idx_lhs[tensor_idx + s] = state_r_no # I
                    idx_rhs[tensor_idx + s] = state_no # J

                    idx_lhs = tuple(idx_lhs)
                    idx_rhs = tuple(idx_rhs)

                    #print(idx_lhs)
                    #print(idx_rhs)
                    #print('(', i, j, ')')
                    #print(parity * ke_int[i,j])
                    #print('\n')

                    sigma[idx_lhs] += parity * C[idx_rhs] * ke_int[i, j]

    # Coulomb interactions between
    for symb1, symb2 in itertools.combinations(particles, 2):
        particle1 = particles[symb1]
        particle2 = particles[symb2]

        fbi1 = particle1['full_basis_idx']
        obtl1 = particle1['no_spatial_orbitals']

        fbi2 = particle2['full_basis_idx']
        obtl2 = particle2['no_spatial_orbitals']

        tensor_idx1 = particle1['tensor_idx']
        tensor_idx2 = particle2['tensor_idx']

        spin1 = particle1['properties']['spin']
        spin2 = particle2['properties']['spin']

        cmb_int = full_cmb_int[fbi1:fbi1+obtl1, fbi1:fbi1+obtl1, fbi2:fbi2+obtl2, fbi2:fbi2+obtl2]

        D = np.zeros(tuple(string_mtx_shape) + (obtl2, obtl2))
        E = np.zeros(tuple(string_mtx_shape) + (obtl1, obtl1))

        for s2 in range(spin2):
            for state_no, (state, states_r) in enumerate(zip(particle2['states'][s2], particle2['states_r'][s2])):
                for state_r_no, a, b, parity in states_r:
                        #print('~~~~~~~~')
                        #print(state_no, state_r_no, '(', i, j, ')', parity)
                    idx_lhs = [slice(None)] * D.ndim
                    idx_rhs = [slice(None)] * C.ndim

                    idx_lhs[tensor_idx2 + s2] = state_no
                    idx_rhs[tensor_idx2 + s2] = state_r_no

                    idx_lhs[-2] = a
                    idx_lhs[-1] = b

                    idx_lhs = tuple(idx_lhs)
                    idx_rhs = tuple(idx_rhs)

                    D[idx_lhs] += C[idx_rhs] * parity

        E = np.tensordot(D, cmb_int, axes=([-2, -1], [-2, -1]))

        """
        for i, j, a, b in np.ndindex((obtl1, obtl1, obtl2, obtl2)):
            idx_lhs = [slice(None)] * E.ndim
            idx_rhs = [slice(None)] * D.ndim

            idx_lhs[-2] = i
            idx_lhs[-1] = j

            idx_rhs[-2] = a
            idx_rhs[-1] = b

            idx_lhs = tuple(idx_lhs)
            idx_rhs = tuple(idx_rhs)

            E[idx_lhs] += D[idx_rhs] * full_cmb_int[i+fbi1, j+fbi1, a+fbi2, b+fbi2]
        """

        for s1 in range(spin1):
            for state_no, (state, states_r) in enumerate(zip(particle1['states'][s1], particle1['states_r'][s1])):
                for state_r_no, i, j, parity in states_r:
                    idx_lhs = [slice(None)] * sigma.ndim
                    idx_rhs = [slice(None)] * E.ndim

                    idx_lhs[tensor_idx1 + s1] = state_r_no
                    idx_rhs[tensor_idx1 + s1] = state_no

                    idx_rhs[-2] = i
                    idx_rhs[-1] = j

                    idx_lhs = tuple(idx_lhs)
                    idx_rhs = tuple(idx_rhs)

                    sigma[idx_lhs] += E[idx_rhs] * parity

    return(sigma.reshape(total_states, order='F'))


Using CPU
converged SCF energy = -1.128506644129
[[[[ 4.36333265e+00  3.58188686e-01  5.20223040e-13 ...  1.53979434e-14
     2.70089309e-14 -8.44563980e-01]
   [ 3.58188686e-01  2.28996462e+00 -4.96576819e-13 ... -2.22902224e-14
    -3.86046639e-14 -7.98292082e-01]
   [ 5.20223040e-13 -4.96576819e-13  2.82750336e+00 ... -7.00538922e-03
    -1.14164848e-02 -3.95884744e-14]
   ...
   [ 1.53979434e-14 -2.22902224e-14 -7.00538922e-03 ... -7.06538878e-01
     2.17515867e-12 -1.67335903e-15]
   [ 2.70089309e-14 -3.86046639e-14 -1.14164848e-02 ...  2.17516934e-12
    -7.06538878e-01 -1.36920318e-15]
   [-8.44563980e-01 -7.98292082e-01 -3.95884744e-14 ... -1.67335903e-15
    -1.36920318e-15 -1.13301869e+00]]

  [[ 3.58188692e-01  9.29010444e-02  2.74078254e-14 ...  3.62872003e-17
     1.41998641e-16 -9.10883756e-02]
   [ 9.29010444e-02  2.46147134e-01 -5.31423891e-14 ...  2.40813353e-17
    -1.14790286e-17 -9.38581901e-02]
   [ 2.74078254e-14 -5.31423891e-14  3.24922466e-01 ... -4.59396012e-0

In [143]:
from scipy.sparse.linalg import LinearOperator
A = LinearOperator(shape=(total_states, total_states), matvec=matvec, dtype=float)

a_eigvals, a_eigvecs = sp.sparse.linalg.eigsh(A, k=2, which='SA', tol=1e-4, maxiter=250)

matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec called
matvec

In [133]:
%%sql
# cc-pVTZ + DZSPN [-1.12231753 -1.09135489]
# cc-pVDZ + DZSPDN [-1.11165122 -1.09684909]

Exception: Variable Name is not chosen

In [140]:
print(total_states)

78000


In [144]:
print(a_eigvals)

[-1.11165122 -1.09684909]


In [17]:
print(get_addr(particles['H']['states'][0][4], particles['H']['addr_arrays'][0]))

4


In [145]:
(a_eigvals[1]-a_eigvals[0])*219474.63

np.float64(3248.6923200324536)

In [69]:
print(full_transform[0])

print(particles['H']['transform'])

[ 3.48189572e+00 -2.77022267e+00 -1.26237716e-07  1.32335894e-07
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
[[ 3.48189572e+00 -2.77022267e+00 -1.26237716e-07  1.32335894e-07]
 [-2.77022267e+00  3.48189572e+00  1.32335895e-07 -1.39310009e-07]
 [-1.26237716e-07  1.32335894e-07  3.48189572e+00 -2.77022267e+00]
 [ 1.32335894e-07 -1.39310009e-07 -2.77022267e+00  3.48189572e+00]]


In [61]:
p = 'H'
spin = 0
idx = 1

print(particles[p]['states'][spin])

print(particles[p]['states'][spin][idx])
print(particles[p]['states_r'][spin][idx])

[[0, 1], [0, 2], [0, 3], [1, 2], [1, 3], [2, 3]]
[0, 2]
[(np.int64(3), 0, 1, 1), (np.int64(5), 0, 3, -1), (np.int64(0), 2, 1, 1), (np.int64(2), 2, 3, 1)]


In [61]:
print(len(full_basis))

8


In [51]:
particles['H']['addr_arrays'][1].shape

(1, 0)

In [87]:
particles['H']['addr_arrays'][0]

array([[0, 2, 3, 0],
       [0, 0, 1, 2]])

In [73]:
print(len(full_basis))
print(full_transform.shape)

8
(8, 8)


In [10]:
print(particles['H']['states'][1])

[[]]


In [58]:
print(tuple(itertools.chain.from_iterable([particles[symb]['basis'] for symb in particles])))

(<gbasis.contractions.GeneralizedContractionShell object at 0x7e0880474740>, <gbasis.contractions.GeneralizedContractionShell object at 0x7e08807e4a40>, <gbasis.contractions.GeneralizedContractionShell object at 0x7e0880580c50>, <gbasis.contractions.GeneralizedContractionShell object at 0x7e088052aba0>, <gbasis.wrappers.from_pyscf.<locals>.PyscfShell object at 0x7e08805b7e30>, <gbasis.wrappers.from_pyscf.<locals>.PyscfShell object at 0x7e08807e5a00>, <gbasis.wrappers.from_pyscf.<locals>.PyscfShell object at 0x7e0880589190>, <gbasis.wrappers.from_pyscf.<locals>.PyscfShell object at 0x7e088995aae0>)


In [93]:
for state_no, (state, states_r) in enumerate(zip(particles['H']['states'][0], particles['H']['states_r'][0])):
    print(state_no, states_r)
    for state_r_no, i, j, parity in states_r:
    print(state)
    print('\n')

0 [(np.int64(3), 0, 2, -1), (np.int64(4), 0, 3, -1), (np.int64(1), 1, 2, 1), (np.int64(2), 1, 3, 1)]
3
4
1
2
[0, 1]


1 [(np.int64(3), 0, 1, 1), (np.int64(5), 0, 3, -1), (np.int64(0), 2, 1, 1), (np.int64(2), 2, 3, 1)]
3
5
0
2
[0, 2]


2 [(np.int64(4), 0, 1, 1), (np.int64(5), 0, 2, 1), (np.int64(0), 3, 1, 1), (np.int64(1), 3, 2, 1)]
4
5
0
1
[0, 3]


3 [(np.int64(1), 1, 0, 1), (np.int64(5), 1, 3, -1), (np.int64(0), 2, 0, -1), (np.int64(4), 2, 3, 1)]
1
5
0
4
[1, 2]


4 [(np.int64(2), 1, 0, 1), (np.int64(5), 1, 2, 1), (np.int64(0), 3, 0, -1), (np.int64(3), 3, 2, 1)]
2
5
0
3
[1, 3]


5 [(np.int64(2), 2, 0, 1), (np.int64(4), 2, 1, 1), (np.int64(1), 3, 0, -1), (np.int64(3), 3, 1, -1)]
2
4
1
3
[2, 3]




In [77]:
print(particles['H'].keys())
print(particles['H']['counts'])
print(particles['H']['no_spatial_orbitals'])

dict_keys(['coords', 'count', 'basis', 'transform', 'no_spatial_orbitals', 'properties', 'idx', 'no_spin_orbitals', 'occs', 'states', 'addr_arrays', 'bases', 'counts', 'base', 'states_r'])
[6, 1]
4


In [50]:
print(particles['e']['idx'])

1


dict_keys(['coords', 'count', 'basis', 'transform', 'no_spatial_orbitals', 'properties', 'idx', 'no_spin_orbitals', 'occs', 'states', 'addr_arrays', 'bases', 'counts', 'tensor_idx', 'base', 'states_r'])
